In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install mediapipe==0.10.13

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# BLOCK 0: CONFIG
# ════════════════════════════════════════════════════════════════════════════════

import os
import subprocess
import cv2
import numpy as np
import mediapipe as mp
from tqdm.notebook import tqdm

INPUT_BASE  = "/content/drive/MyDrive/KArSL-502/02/train"            # .7z files
EXTRACT_DIR = "/content/drive/MyDrive/karsl_02_extracted_2"   # extracted frames on Drive
OUTPUT_BASE = "/content/drive/MyDrive/karsl_work_02_fixed"  # final .npy on Drive
SIGNER      = "02"
SPLITS      = ["train", "test"]

os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUTPUT_BASE, exist_ok=True)

mp_holistic = mp.solutions.holistic

def adjust_landmarks(arr, center):
    arr_reshaped    = arr.reshape(-1, 3)
    center_repeated = np.tile(center, (len(arr_reshaped), 1))
    arr_adjusted    = arr_reshaped - center_repeated
    return arr_adjusted.reshape(-1)

def extract_keypoints(results):
    pose = np.array([[r.x, r.y, r.z] for r in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*3)
    lh   = np.array([[r.x, r.y, r.z] for r in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh   = np.array([[r.x, r.y, r.z] for r in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    pose_adj = adjust_landmarks(pose, pose[:3])
    lh_adj   = adjust_landmarks(lh,   lh[:3])
    rh_adj   = adjust_landmarks(rh,   rh[:3])
    return pose_adj, lh_adj, rh_adj

print("✓ Config loaded")
print(f"Input:   {INPUT_BASE}")
print(f"Extract: {EXTRACT_DIR}")
print(f"Output:  {OUTPUT_BASE}")

✓ Config loaded
Input:   /content/drive/MyDrive/KArSL-502/02/train
Extract: /content/drive/MyDrive/karsl_02_extracted_2
Output:  /content/drive/MyDrive/karsl_work_02_fixed


In [ ]:
import os

# Try to access signer 02 data through shortcut
path_02 = "/content/drive/MyDrive/KArSL-502/02"

print(f"Exists: {os.path.exists(path_02)}")
print(f"Is dir: {os.path.isdir(path_02)}")

try:
    contents = os.listdir(path_02)
    print(f"Contents ({len(contents)}): {contents[:10]}")

    # Go deeper
    if contents:
        first = os.path.join(path_02, contents[0])
        print(f"\n{contents[0]}/:")
        inner = os.listdir(first)
        print(f"  {inner[:5]}")
except Exception as e:
    print(f"Error: {e}")

Exists: True
Is dir: True
Contents (2): ['test', 'train']

test/:
  ['0001-0070.7z', '0071-0170.7z', '0171-0502.7z']


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# BLOCK 1: INSTALL 7Z AND EXPLORE INPUT
# ════════════════════════════════════════════════════════════════════════════════

subprocess.run(["apt-get", "install", "-y", "p7zip-full"], capture_output=True)
print("✓ p7zip installed")


print(f"\nContents of {INPUT_BASE}:")
for item in sorted(os.listdir(INPUT_BASE)):
    item_path = os.path.join(INPUT_BASE, item)
    size_mb   = os.path.getsize(item_path) / (1024*1024)
    print(f"  {item} ({size_mb:.1f} MB)")

✓ p7zip installed

Contents of /content/drive/MyDrive/KArSL-502/02/train:
  0001-0070.7z (328.4 MB)
  0071-0170.7z (1226.5 MB)
  0171-0502.7z (3492.4 MB)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# BLOCK 2: EXTRACT .7Z TO DRIVE — checkpointed
# ════════════════════════════════════════════════════════════════════════════════

zip_files = sorted([f for f in os.listdir(INPUT_BASE) if f.endswith(".7z")])
print(f"Found {len(zip_files)} .7z files: {zip_files}")

for zip_file in tqdm(zip_files, desc="Extracting"):
    zip_path    = os.path.join(INPUT_BASE, zip_file)
    folder_name = zip_file.replace(".7z", "")
    extract_to  = os.path.join(EXTRACT_DIR, folder_name)

    # Skip if already extracted
    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f"  ✓ Already extracted: {zip_file}")
        continue

    print(f"  Extracting: {zip_file} → {extract_to}")
    result = subprocess.run(
        ["7z", "x", zip_path, f"-o{EXTRACT_DIR}", "-y"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✓ Done: {zip_file}")
    else:
        print(f"  ✗ Failed: {zip_file}\n{result.stderr[:300]}")

Found 3 .7z files: ['0001-0070.7z', '0071-0170.7z', '0171-0502.7z']


Extracting:   0%|          | 0/3 [00:00<?, ?it/s]

  Extracting: 0001-0070.7z → /content/drive/MyDrive/karsl_02_extracted_2/0001-0070
  ✓ Done: 0001-0070.7z
  Extracting: 0071-0170.7z → /content/drive/MyDrive/karsl_02_extracted_2/0071-0170


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# BLOCK 3: EXPLORE EXTRACTED STRUCTURE
# ════════════════════════════════════════════════════════════════════════════════

print(f"Extracted structure: {EXTRACT_DIR}")
for item in sorted(os.listdir(EXTRACT_DIR)):
    item_path = os.path.join(EXTRACT_DIR, item)
    if not os.path.isdir(item_path): continue
    children = sorted(os.listdir(item_path))
    print(f"\n{item}/ ({len(children)} items) → {children[:3]}")

    if children:
        l2 = os.path.join(item_path, children[0])
        if os.path.isdir(l2):
            l2c = sorted(os.listdir(l2))
            print(f"  {children[0]}/ ({len(l2c)}) → {l2c[:3]}")
            if l2c:
                l3 = os.path.join(l2, l2c[0])
                if os.path.isdir(l3):
                    l3c = sorted(os.listdir(l3))
                    print(f"    {l2c[0]}/ ({len(l3c)}) → {l3c[:3]}")
                    if l3c:
                        l4 = os.path.join(l3, l3c[0])
                        if os.path.isdir(l4):
                            l4c = sorted(os.listdir(l4))
                            print(f"      {l3c[0]}/ ({len(l4c)} files) → {l4c[:2]}")

In [ ]:
import os

# Update with your actual path
BASE = "/content/drive/MyDrive/karsl_02"

for level1 in sorted(os.listdir(BASE))[:3]:
    l1_path = os.path.join(BASE, level1)
    if not os.path.isdir(l1_path): continue
    print(f"\n{level1}/")

    for level2 in sorted(os.listdir(l1_path))[:3]:
        l2_path = os.path.join(l1_path, level2)
        if not os.path.isdir(l2_path): continue
        print(f"  {level2}/")

        for level3 in sorted(os.listdir(l2_path))[:3]:
            l3_path = os.path.join(l2_path, level3)
            if not os.path.isdir(l3_path): continue
            print(f"    {level3}/")

            for level4 in sorted(os.listdir(l3_path))[:3]:
                l4_path = os.path.join(l3_path, level4)
                if not os.path.isdir(l4_path): continue
                files = os.listdir(l4_path)
                print(f"      {level4}/ ({len(files)} files | sample: {files[0] if files else 'empty'})")


0001-0070/
  0001/
  0002/
  0003/

0071-0170/
  0071/
    03_02_0071_(12_12_16_15_54_54)_c/
    03_02_0071_(12_12_16_15_54_58)_c/
    03_02_0071_(12_12_16_15_55_01)_c/
  0072/
    03_02_0072_(12_12_16_15_57_50)_c/
    03_02_0072_(12_12_16_15_57_55)_c/
    03_02_0072_(12_12_16_15_58_00)_c/
  0073/
    03_02_0073_(12_12_16_16_01_20)_c/
    03_02_0073_(12_12_16_16_01_24)_c/
    03_02_0073_(12_12_16_16_01_29)_c/

0171-0502/
  F11/
    0255/
      03_02_0255_(14_11_17_10_46_51)_c/ (50 files | sample: 03_02_0255_(14_11_17_10_46_51)_c_0002.jpg)
      03_02_0255_(14_11_17_10_46_54)_c/ (44 files | sample: 03_02_0255_(14_11_17_10_46_54)_c_0001.jpg)
      03_02_0255_(14_11_17_10_47_01)_c/ (45 files | sample: 03_02_0255_(14_11_17_10_47_01)_c_0002.jpg)
    0483/
      03_02_0483_(15_11_17_17_51_31)_c/ (44 files | sample: 03_02_0483_(15_11_17_17_51_31)_c_0007.jpg)
      03_02_0483_(15_11_17_17_55_35)_c/ (61 files | sample: 03_02_0483_(15_11_17_17_55_35)_c_0005.jpg)
      03_02_0483_(15_11_17_17_55

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# COLLECT VIDEOS — handles both simple and F-subfolder structures
# ════════════════════════════════════════════════════════════════════════════════
INPUT_BASE = "/content/drive/MyDrive/karsl_02"

def is_word_folder(name):
    """Word folders are 4-digit numbers like 0001, 0071, 0255."""
    return name.isdigit() and len(name) == 4

def is_video_folder(name):
    """Video folders contain timestamp pattern like 03_02_0001_(...)_c."""
    return "_c" in name

def collect_videos(split_path):
    """
    Handles two structures:
    1. part_folder/word_folder/video_folder/frames  (0001-0070, 0071-0170)
    2. part_folder/F_folder/word_folder/video_folder/frames  (0171-0502)
    """
    all_videos = []

    for part_folder in sorted(os.listdir(split_path)):
        part_path = os.path.join(split_path, part_folder)
        if not os.path.isdir(part_path):
            continue

        for item in sorted(os.listdir(part_path)):
            item_path = os.path.join(part_path, item)
            if not os.path.isdir(item_path):
                continue

            if is_word_folder(item):
                # Structure 1: part/word/video/frames
                word_folder = item
                word_path   = item_path
                for video_folder in sorted(os.listdir(word_path)):
                    video_path = os.path.join(word_path, video_folder)
                    if not os.path.isdir(video_path):
                        continue
                    frames = [f for f in os.listdir(video_path)
                              if f.lower().endswith(('.jpg','.jpeg','.png'))]
                    if len(frames) > 0:
                        all_videos.append((word_folder, video_folder, video_path))

            else:
                # Structure 2: part/F_folder/word/video/frames
                f_folder  = item
                f_path    = item_path
                for word_folder in sorted(os.listdir(f_path)):
                    word_path = os.path.join(f_path, word_folder)
                    if not os.path.isdir(word_path) or not is_word_folder(word_folder):
                        continue
                    for video_folder in sorted(os.listdir(word_path)):
                        video_path = os.path.join(word_path, video_folder)
                        if not os.path.isdir(video_path):
                            continue
                        frames = [f for f in os.listdir(video_path)
                                  if f.lower().endswith(('.jpg','.jpeg','.png'))]
                        if len(frames) > 0:
                            all_videos.append((word_folder, video_folder, video_path))

    words = set(v[0] for v in all_videos)
    print(f"  Collected: {len(all_videos)} videos | {len(words)} words")
    print(f"  Word range: {min(words)} - {max(words)}")
    return all_videos


# Test collection
print("Testing collection on train...")
split_path   = INPUT_BASE  # update if needed
train_videos = collect_videos(split_path)

Testing collection on train...
  Collected: 20761 videos | 490 words
  Word range: 0007 - 0502


In [ ]:
import pickle

# Save train_videos list to Drive
train_videos_path = "/content/drive/MyDrive/karsl_02_train_videos.pkl"

with open(train_videos_path, "wb") as f:
    pickle.dump(train_videos, f)

print(f"✓ Saved {len(train_videos)} videos to {train_videos_path}")

# To load after disconnect:
# with open(train_videos_path, "rb") as f:
#     train_videos = pickle.load(f)
# print(f"Loaded {len(train_videos)} videos")

✓ Saved 20761 videos to /content/drive/MyDrive/karsl_02_train_videos.pkl


In [ ]:
import pickle

# Save train_videos list to Drive
train_videos_path = "/content/drive/MyDrive/karsl_02_train_videos.pkl"

with open(train_videos_path, "rb") as f:
     train_videos = pickle.load(f)
print(f"Loaded {len(train_videos)} videos")

Loaded 20761 videos


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# RUN MEDIAPIPE EXTRACTION
# ════════════════════════════════════════════════════════════════════════════════

import cv2
import numpy as np
import mediapipe as mp
from tqdm.notebook import tqdm

mp_holistic = mp.solutions.holistic

def adjust_landmarks(arr, center):
    arr_reshaped    = arr.reshape(-1, 3)
    center_repeated = np.tile(center, (len(arr_reshaped), 1))
    arr_adjusted    = arr_reshaped - center_repeated
    return arr_adjusted.reshape(-1)

def extract_keypoints(results):
    pose = np.array([[r.x, r.y, r.z] for r in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*3)
    lh   = np.array([[r.x, r.y, r.z] for r in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh   = np.array([[r.x, r.y, r.z] for r in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    pose_adj = adjust_landmarks(pose, pose[:3])
    lh_adj   = adjust_landmarks(lh,   lh[:3])
    rh_adj   = adjust_landmarks(rh,   rh[:3])
    return pose_adj, lh_adj, rh_adj


def run_extraction(all_videos, signer, split, output_base):
    success = 0; skipped = 0; failed = 0

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        pbar = tqdm(all_videos, desc=f"Signer {signer} | {split}")

        for word_folder, video_folder, video_path in pbar:
            out_word  = os.path.join(output_base, signer, split, word_folder)
            pose_dir  = os.path.join(out_word, "pose_keypoints")
            lh_dir    = os.path.join(out_word, "lh_keypoints")
            rh_dir    = os.path.join(out_word, "rh_keypoints")
            pose_file = os.path.join(pose_dir, video_folder + ".npy")

            # Checkpoint
            if os.path.exists(pose_file):
                skipped += 1
                continue

            os.makedirs(pose_dir, exist_ok=True)
            os.makedirs(lh_dir,   exist_ok=True)
            os.makedirs(rh_dir,   exist_ok=True)

            pose_kps, lh_kps, rh_kps = [], [], []
            frame_files = sorted([
                f for f in os.listdir(video_path)
                if f.lower().endswith(('.jpg','.jpeg','.png'))
            ])

            for frame_file in frame_files:
                frame = cv2.imread(os.path.join(video_path, frame_file))
                if frame is None:
                    continue
                image   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = holistic.process(image)
                p, lh, rh = extract_keypoints(results)
                pose_kps.append(p)
                lh_kps.append(lh)
                rh_kps.append(rh)

            if len(pose_kps) == 0:
                failed += 1
                continue

            np.save(pose_file,
                    np.array(pose_kps))
            np.save(os.path.join(lh_dir, video_folder+".npy"),
                    np.array(lh_kps))
            np.save(os.path.join(rh_dir, video_folder+".npy"),
                    np.array(rh_kps))
            success += 1

            if (success + skipped) % 200 == 0:
                pbar.set_description(
                    f"done={success} skip={skipped} fail={failed}"
                )

    print(f"\n✓ {split}: extracted={success} | skipped={skipped} | failed={failed}")

In [ ]:
run_extraction(train_videos, SIGNER, "train", OUTPUT_BASE)

Signer 02 | train:   0%|          | 0/20761 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VERIFY — train only
# ════════════════════════════════════════════════════════════════════════════════

split    = "train"
split_dir = os.path.join(OUTPUT_BASE, SIGNER, split)

if not os.path.exists(split_dir):
    print(f"✗ {split}: NOT FOUND")
else:
    total = 0; non_empty = 0; words = 0
    for word in os.listdir(split_dir):
        pose_dir = os.path.join(split_dir, word, "pose_keypoints")
        if not os.path.exists(pose_dir): continue
        words += 1
        for f in os.listdir(pose_dir):
            if not f.endswith(".npy"): continue
            total += 1
            data   = np.load(os.path.join(pose_dir, f))
            if data.size > 0: non_empty += 1

    pct    = non_empty/total*100 if total > 0 else 0
    status = "✓" if pct > 90 else "✗"
    print(f"{status} Signer {SIGNER} | {split}: "
          f"{non_empty}/{total} valid ({pct:.1f}%) | {words} words")